In [ ]:
# ============================
# L-BFGS ARREGLADO PARA obj_function(image, V_obs, weights, reg_lambda, reg_func)
# ============================

import time
import numpy as np

try:
    import cupy as cp
    xp = cp
    USING_CUPY = True
except:
    xp = np
    USING_CUPY = False

Array = xp.ndarray


# -------------------------------------
# Helpers
# -------------------------------------
def _as_xp(arr):
    if USING_CUPY and isinstance(arr, np.ndarray):
        return cp.asarray(arr)
    return arr

def _norm(x):
    return float(xp.linalg.norm(x.ravel()))

def _dot(a, b):
    return float(xp.vdot(a.ravel(), b.ravel()))


# -------------------------------------
# LBFGS Memory
# -------------------------------------
class LBFGSState:
    def __init__(self, m: int):
        self.m = m
        self.s_list = []
        self.y_list = []
        self.rho_list = []

    def add_pair(self, s, y, min_sy=1e-12):
        sy = _dot(s, y)
        if sy <= min_sy:
            return False
        
        rho = 1.0 / sy

        if len(self.s_list) == self.m:
            self.s_list.pop(0)
            self.y_list.pop(0)
            self.rho_list.pop(0)

        self.s_list.append(s)
        self.y_list.append(y)
        self.rho_list.append(rho)
        return True


# -------------------------------------
# TWO LOOP RECURSION
# -------------------------------------
def lbfgs_two_loop(grad, state, eps=1e-16):
    if len(state.s_list) == 0:
        return -grad

    q = grad.copy()
    alpha = []

    # backward loop
    for s, y, rho in zip(reversed(state.s_list), reversed(state.y_list), reversed(state.rho_list)):
        a = rho * _dot(s, q)
        alpha.append(a)
        q = q - a * y

    # scaling
    s_last = state.s_list[-1]
    y_last = state.y_list[-1]
    gamma = _dot(s_last, y_last) / (_dot(y_last, y_last) + eps)
    r = gamma * q

    # forward loop
    for s, y, rho, a in zip(state.s_list, state.y_list, state.rho_list, reversed(alpha)):
        beta = rho * _dot(y, r)
        r = r + s * (a - beta)

    return -r


# -------------------------------------
# ARMIJO LINE SEARCH
# -------------------------------------
def armijo_backtracking(
    image, direction, obj_function, args,
    current_cost, grad,
    c1=1e-4, rho=0.5, alpha0=1.0,
    alpha_min=1e-12, max_iter=50
):
    g_dot_d = _dot(grad, direction)
    if g_dot_d >= 0:
        return 0.0, current_cost, "non_descent"

    alpha = alpha0

    for _ in range(max_iter):
        new_img = image + alpha * direction
        cost_new, _ = obj_function(new_img, *args)

        if cost_new <= current_cost + c1 * alpha * g_dot_d:
            return float(alpha), float(cost_new), "ok"

        alpha *= rho
        if alpha < alpha_min:
            break

    return float(alpha), float(cost_new), "min_alpha"


# -------------------------------------
# OPTIMIZER
# -------------------------------------
def lbfgs_optimize_3(
    obj_function,
    args,
    x0,
    m=10,
    max_iter=100,
    gtol=1e-6,
    ftol=1e-12,
    verbose=True
):
    global xp, USING_CUPY

    x = _as_xp(x0)

    state = LBFGSState(m=m)

    cost, grad = obj_function(x, *args)
    cost = float(cost)
    grad = _as_xp(grad)

    cost_history = [cost]
    t_start = time.time()

    if verbose:
        print(f"[LBFGS] it=0 cost={cost:.6e} ||g||={_norm(grad):.3e}")

    # -----------------------
    # MAIN LOOP
    # -----------------------
    for k in range(1, max_iter+1):

        gnorm = _norm(grad)
        if gnorm <= gtol:
            if verbose:
                print(f"[LBFGS] Converged (grad) at iter {k}")
            break

        direction = lbfgs_two_loop(grad, state)

        # direction must be descent
        if _dot(grad, direction) >= 0:
            direction = -grad

        # line search
        alpha, cost_new, status = armijo_backtracking(
            x, direction, obj_function, args, cost, grad
        )

        # fallback if needed
        if status in ["non_descent", "min_alpha"]:
            direction = -grad
            alpha = 1e-3
            x_next = x + alpha * direction
            cost_new, grad_new = obj_function(x_next, *args)
        else:
            x_next = x + alpha * direction
            cost_new, grad_new = obj_function(x_next, *args)

        cost_new = float(cost_new)
        grad_new = _as_xp(grad_new)

        # update memory
        s = x_next - x
        y = grad_new - grad
        state.add_pair(s, y)

        # update step
        x = x_next
        cost_prev = cost
        cost = cost_new
        grad = grad_new
        cost_history.append(cost)

        if verbose:
            print(f"[LBFGS] it={k} cost={cost:.6e} ||g||={_norm(grad):.3e} alpha={alpha:.2e}")

        # stopping by function decrease
        denom = abs(cost) + abs(cost_prev) + 1e-16
        rel_change = 2 * abs(cost - cost_prev) / denom
        if rel_change <= ftol:
            if verbose:
                print(f"[LBFGS] Converged (Δf small) at iter {k}")
            break

    total_time = time.time() - t_start

    info = {
        "cost_history": cost_history,
        "niter": len(cost_history)-1,
        "time": total_time
    }

    return x, info


In [ ]:
# # ===========================================
# #   L-BFGS COMPACTO (CPU / GPU)
# #   Modificado para usar obj_function(image, V_obs, weights, reg_lambda, reg_func)
# # ===========================================

# import time
# import numpy as np

# # Try CUPY
# try:
#     import cupy as cp
#     xp = cp
#     USING_CUPY = True
# except:
#     xp = np
#     USING_CUPY = False

# Array = xp.ndarray

# def _as_xp(arr):
#     if USING_CUPY and isinstance(arr, np.ndarray):
#         return cp.asarray(arr)
#     return arr

# def _norm(x: Array) -> float:
#     return float(xp.linalg.norm(x.ravel()))

# def _dot(a: Array, b: Array) -> float:
#     return float(xp.vdot(a.ravel(), b.ravel()))

# # =====================================================
# #   MEMORY STORAGE
# # =====================================================
# class LBFGSState:
#     def __init__(self, m: int):
#         self.m = m
#         self.s_list = []
#         self.y_list = []
#         self.rho_list = []

#     def add_pair(self, s, y, min_sy=1e-12):
#         sy = _dot(s, y)
#         if sy <= min_sy:
#             return False
#         rho = 1.0 / sy

#         if len(self.s_list) == self.m:
#             self.s_list.pop(0)
#             self.y_list.pop(0)
#             self.rho_list.pop(0)

#         self.s_list.append(s)
#         self.y_list.append(y)
#         self.rho_list.append(rho)
#         return True

# # =====================================================
# #   TWO LOOP RECURSION
# # =====================================================
# def lbfgs_two_loop(grad, state: LBFGSState, eps=1e-16):
#     if len(state.s_list) == 0:
#         return -grad

#     q = grad.copy()
#     alpha = []

#     # First loop — backward
#     for s, y, rho in zip(reversed(state.s_list), reversed(state.y_list), reversed(state.rho_list)):
#         a = rho * _dot(s, q)
#         alpha.append(a)
#         q = q - a * y

#     # Scaling
#     s_last = state.s_list[-1]
#     y_last = state.y_list[-1]
#     gamma = _dot(s_last, y_last) / (_dot(y_last, y_last) + eps)
#     r = gamma * q

#     # Second loop — forward
#     for s, y, rho, a in zip(state.s_list, state.y_list, state.rho_list, reversed(alpha)):
#         beta = rho * _dot(y, r)
#         r = r + s * (a - beta)

#     return -r


# # =====================================================
# #   ARMIJO LINE SEARCH
# # =====================================================
# def armijo_backtracking(
#     image, direction, obj_function, args,
#     current_cost, grad,
#     c1=1e-4, rho=0.5, alpha0=1.0,
#     alpha_min=1e-12, max_iter=50
# ):
#     g_dot_d = _dot(grad, direction)
#     if g_dot_d >= 0:
#         return 0.0, current_cost, "non_descent"

#     alpha = alpha0
#     for _ in range(max_iter):
#         new_img = image + alpha * direction
#         cost_new, _ = obj_function(new_img, *args)

#         if cost_new <= current_cost + c1 * alpha * g_dot_d:
#             return float(alpha), float(cost_new), "ok"

#         alpha *= rho
#         if alpha < alpha_min:
#             break

#     return float(alpha), float(cost_new), "min_alpha"


# # =====================================================
# #              L-BFGS OPTIMIZER
# # =====================================================
# def lbfgs_optimize_2(
#     obj_function,        # <===== tu función obj_function(image, V_obs, weights, reg_lambda, reg_func)
#     x0,
#     args,               # (V_obs, weights, reg_lambda, reg_func)
#     m=10,
#     max_iter=100,
#     gtol=1e-6,
#     ftol=1e-12,
#     verbose=True
# ):
#     global xp, USING_CUPY

#     # Move x0 to GPU/CPU as needed
#     x = _as_xp(x0)

#     # Initial evaluation
#     cost, grad = obj_function(x, *args)
#     grad = _as_xp(grad)
#     cost = float(cost)

#     state = LBFGSState(m=m)
#     cost_history = [cost]
#     t0 = time.time()

#     if verbose:
#         print(f"[LBFGS] it=0 cost={cost:.6e} ||g||={_norm(grad):.3e}")

#     # ================================
#     #      MAIN LOOP
#     # ================================
#     for it in range(1, max_iter + 1):

#         gnorm = _norm(grad)
#         if gnorm <= gtol:
#             if verbose:
#                 print(f"[LBFGS] Converged (||g||={gnorm:.3e}) at it={it}")
#             break

#         # Compute direction
#         direction = lbfgs_two_loop(grad, state)

#         # Safety: ensure descent
#         if _dot(grad, direction) >= 0:
#             direction = -grad

#         # Line search
#         alpha, cost_new, status = armijo_backtracking(
#             x, direction, obj_function, args,
#             cost, grad
#         )

#         if status == "non_descent":
#             # fallback to gradient descent
#             direction = -grad
#             alpha = 1e-3
#             x_new = x + alpha * direction
#             cost_new, grad_new = obj_function(x_new, *args)

#             x = x_new
#             grad = _as_xp(grad_new)
#             cost = float(cost_new)
#             cost_history.append(cost)

#             if verbose:
#                 print(f"[LBFGS] fallback step α={alpha:.1e} cost={cost:.6e}")
#             continue

#         # Standard accepted step
#         x_new = x + alpha * direction
#         cost_prev = cost
#         grad_prev = grad

#         cost_new, grad_new = obj_function(x_new, *args)
#         cost_new = float(cost_new)
#         grad_new = _as_xp(grad_new)

#         # Update memory
#         s = x_new - x
#         y = grad_new - grad_prev
#         state.add_pair(s, y)

#         x = x_new
#         grad = grad_new
#         cost = cost_new
#         cost_history.append(cost)

#         if verbose:
#             print(f"[LBFGS] it={it} cost={cost:.6e} ||g||={_norm(grad):.3e}  α={alpha:.2e}")

#         # Check relative decrease in f
#         denom = abs(cost) + abs(cost_prev) + 1e-16
#         rel = 2 * abs(cost - cost_prev) / denom
#         if rel <= ftol:
#             if verbose:
#                 print(f"[LBFGS] Converged (Δf small={rel:.3e}) at it={it}")
#             break

#     # END LOOP
#     total_time = time.time() - t0
#     info = {
#         "cost_history": cost_history,
#         "niter": len(cost_history) - 1,
#         "time": total_time
#     }

#     return x, info


[LBFGS] it=0 cost=0.000000e+00 ||g||=1.000e+01
[LBFGS] it=1 cost=-2.500000e+01 ||g||=0.000e+00 alpha=5.00e-01
[LBFGS] Converged (grad norm 0.000e+00 <= gtol) at iter 1
time 0.003428220748901367 iters 1
